# Week 01 — Capstone project

I am given eight synthetic black-box functions. These are unknown mathematical functions that accept inputs and return a single output. Our goal is to find the inputs that give the highest possible output for each function.

Every function is a maximisation problem. We won't see the internal workings of the functions, but we’ll be able to observe how they respond to different inputs.

Each function is:

A maximisation task 
Initially represented by ten known data points of increasing dimensionality (from 2D to 8D)

### Summary
Here's how the capstone project works:

- You're maximising eight unknown functions, one query per function per week.
- You choose the ML method, such as random, grid, Bayesian optimisation or manual.
- You submit your inputs in a precise format via the capstone project portal.
- You reflect, revise and iterate over several rounds.
- Your success isn't just the highest value – it’s showing thoughtful, data-driven decision-making in your reflection.






# Week 1 Strategy — GP surrogate + UCB acquisition function

This notebook contains the **Week 1 codebase**:

1. It loads the provided starter data from `.npy` files (`F*_initial_inputs.npy`, `F*_initial_outputs.npy`).
2. Fits a **Gaussian Process (GP)** surrogate per function.
3. Samples many candidate points in $[0,1]^d$.
4. Pick the next query using **UCB**: $\mu(x) + \beta\,\sigma(x)$ (here $\beta=2$).

In [8]:
import numpy as np
import pandas as pd

W1_DIR = 'initial_data'  # change if your folder differs

def load_week0(function_id: int):
    X = np.load(f'{W1_DIR}/function_{function_id}/F{function_id}_initial_inputs.npy')
    y = np.load(f'{W1_DIR}/function_{function_id}/F{function_id}_initial_outputs.npy')
    return X, y

# Sanity check (shapes)
for f in range(1, 9):
    X, y = load_week0(f)
    print(f'F{f}: X={X.shape}, y={y.shape}, y_min={y.min():.4g}, y_max={y.max():.4g}')


F1: X=(10, 2), y=(10,), y_min=-0.003606, y_max=7.711e-16
F2: X=(10, 2), y=(10,), y_min=-0.06562, y_max=0.6112
F3: X=(15, 3), y=(15,), y_min=-0.3989, y_max=-0.03484
F4: X=(30, 4), y=(30,), y_min=-32.63, y_max=-4.026
F5: X=(20, 4), y=(20,), y_min=0.1129, y_max=1089
F6: X=(20, 5), y=(20,), y_min=-2.571, y_max=-0.7143
F7: X=(30, 6), y=(30,), y_min=0.002701, y_max=1.365
F8: X=(40, 8), y=(40,), y_min=5.592, y_max=9.598


## GP + UCB implementation

In [14]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel as C
from sklearn.preprocessing import StandardScaler

def fit_gp(X: np.ndarray, y: np.ndarray, noise: float = 1e-6, seed: int = 42):
    """Fit a GP surrogate on standardised targets (stable across functions)."""
    scaler = StandardScaler()
    y_std = scaler.fit_transform(y.reshape(-1, 1)).ravel()
    kernel = C(1.0, (1e-3, 1e3)) * Matern(length_scale=np.ones(X.shape[1]), nu=2.5) + WhiteKernel(noise_level=noise)
    gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=5, random_state=seed)
    gp.fit(X, y_std)
    return gp

def propose_ucb(gp: GaussianProcessRegressor, d: int, beta: float = 2.0,
                n_candidates: int = 100_000, seed: int = 43) -> np.ndarray:
    """Sample candidates uniformly and return the argmax of UCB."""
    rng = np.random.default_rng(seed)
    Cands = rng.uniform(0.0, 1.0, size=(n_candidates, d))
    mu, std = gp.predict(Cands, return_std=True)
    ucb = mu + beta * std
    return Cands[int(np.argmax(ucb))]

def fmt_portal(x: np.ndarray) -> str:
    return '-'.join(f'{float(v):.6f}' for v in x)


## Generate Week 1 UCB proposals from Week 0 `.npy` data

In [15]:
beta = 2.0
n_candidates = 200_000

ucb_proposals = {}
for f in range(1, 9):
    X0, y0 = load_week0(f)
    gp = fit_gp(X0, y0, seed=100+f)
    x_next = propose_ucb(gp, d=X0.shape[1], beta=beta, n_candidates=n_candidates, seed=200+f)
    ucb_proposals[f] = x_next
    print(f'Function {f}:', fmt_portal(x_next))


/Users/sundarid/opt/anaconda3/lib/python3.9/site-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


Function 1: 0.328862-0.207271
Function 2: 0.964610-0.066806
Function 3: 0.658898-0.129248-0.000014
Function 4: 0.386216-0.410623-0.357460-0.440683


/Users/sundarid/opt/anaconda3/lib/python3.9/site-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


Function 5: 0.900567-0.994070-0.879407-0.881496
Function 6: 0.306275-0.066718-0.672060-0.975918-0.016564


/Users/sundarid/opt/anaconda3/lib/python3.9/site-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/sundarid/opt/anaconda3/lib/python3.9/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 5 of parameter k1__k2__length_scale is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/sundarid/opt/anaconda3/lib/python3.9/site-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 7 of parameter k1__k2__length_scale is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/sundarid/opt/anaconda3/

Function 7: 0.026636-0.389583-0.565497-0.695336-0.419791-0.766605
Function 8: 0.046633-0.263034-0.005694-0.158334-0.974505-0.597902-0.014427-0.531290


In [16]:
week1_submitted = {
  1: np.array([0.759779, 0.804108]),
  2: np.array([0.717915, 0.002064]),
  3: np.array([0.994942, 0.051967, 0.792526]),
  4: np.array([0.404477, 0.413254, 0.303108, 0.434359]),
  5: np.array([0.940282, 0.064909, 0.998637, 0.996528]),
  6: np.array([0.379776, 0.684419, 0.068171, 0.984146, 0.290503]),
  7: np.array([0.015298, 0.590564, 0.658382, 0.751493, 0.425786, 0.776978]),
  8: np.array([0.034492, 0.437681, 0.011459, 0.323393, 0.989664, 0.045780, 0.097175, 0.702465]),
}

for f in range(1, 9):
    print(f'Function {f}:', fmt_portal(week1_submitted[f]))


Function 1: 0.759779-0.804108
Function 2: 0.717915-0.002064
Function 3: 0.994942-0.051967-0.792526
Function 4: 0.404477-0.413254-0.303108-0.434359
Function 5: 0.940282-0.064909-0.998637-0.996528
Function 6: 0.379776-0.684419-0.068171-0.984146-0.290503
Function 7: 0.015298-0.590564-0.658382-0.751493-0.425786-0.776978
Function 8: 0.034492-0.437681-0.011459-0.323393-0.989664-0.045780-0.097175-0.702465


## Notes: Historical record: the exact Week 1 queries that were submitted
As I was experimenting with different hyper parameters here and changed them in the above cell, I did not know exactly what combination gave me the following week 1 inputs and I didn't want to apply a brute force to reproduce those exactly. The above inputs were submitted for week 1
